# DOME Joint Flow Matching Pipeline 探索

这个 notebook 按 non-resample 数据路径一步一步看 joint flow：

1. 读取 joint config，并把 dataset 强制切到 `nuScenesSceneDatasetLidar`。
2. 检查 dataloader 输出的 `input_occs / target_occs / metas`。
3. 从 `metas` 里抽 future trajectory 和 navigation command。
4. 可选：用 OccVAE 把 occupancy 编成 latent。
5. 用一个 tiny `JointDome` 跑通 `JointTrajectoryOccupancyFlowMatching.training_losses()`，确认 shape 和 loss。

## 0. 路径和依赖检查

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'tools' / 'train_joint_flow.py').exists():
    PROJECT_ROOT = Path('/Users/madlineceleste/Desktop/dome-cfm')

sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.executable)

for name in ['torch', 'numpy', 'einops', 'mmengine', 'mmcv', 'timm', 'torchcfm']:
    print(f'{name:10s}', 'OK' if importlib.util.find_spec(name) else 'MISSING')

PROJECT_ROOT = /mnt/data2/whz/dome-cfm
Python = /mnt/data2/congfeng/miniconda3/envs/torchcfm/bin/python
torch      OK
numpy      OK
einops     OK
mmengine   OK
mmcv       OK
timm       OK
torchcfm   OK


## 1. 读取 joint config，并切到 non-resample dataset

当前训练脚本有一个 resample joint config。这里为了看普通 nuScenes 路径，直接在 notebook 里把 `train_dataset_config` 改成 `nuScenesSceneDatasetLidar`。

In [2]:
from mmengine import Config

cfg = Config.fromfile(str(PROJECT_ROOT / 'config' / 'train_dome_joint_flow_resample.py'))

# 强制切到普通 nuScenes dataset，不走 data/resampled_occ。
cfg.train_dataset_config = dict(
    type='nuScenesSceneDatasetLidar',
    data_path='data/nuscenes/',
    imageset='data/nuscenes_infos_train_temporal_v3_scene.pkl',
    offset=0,
    return_len=cfg.return_len_train,
    times=1,
    test_mode=False,
)

# 为了 notebook 快一点，先用小 batch / 单 worker。
cfg.train_loader.batch_size = 2
cfg.train_loader.num_workers = 0

print('sample_method:', cfg.sample.sample_method)
print('train dataset:', cfg.train_dataset_config)
print('return_len_train:', cfg.return_len_train)
print('traj_start_index:', cfg.sample.traj_start_index)
print('traj_len:', cfg.sample.traj_len)
print('traj_dim:', cfg.sample.traj_dim)
print('num_command_modes:', cfg.sample.num_command_modes)

sample_method: joint_flow
train dataset: {'type': 'nuScenesSceneDatasetLidar', 'data_path': 'data/nuscenes/', 'imageset': 'data/nuscenes_infos_train_temporal_v3_scene.pkl', 'offset': 0, 'return_len': 11, 'times': 1, 'test_mode': False}
return_len_train: 11
traj_start_index: 4
traj_len: 6
traj_dim: 2
num_command_modes: 3


## 2. 检查普通 nuScenes 数据文件是否存在

In [3]:
required_paths = [
    PROJECT_ROOT / cfg.train_dataset_config.imageset,
    PROJECT_ROOT / cfg.train_dataset_config.data_path,
]

for p in required_paths:
    print(p, 'EXISTS' if p.exists() else 'MISSING')

CAN_LOAD_REAL_DATA = all(p.exists() for p in required_paths)
print('CAN_LOAD_REAL_DATA =', CAN_LOAD_REAL_DATA)

/mnt/data2/whz/dome-cfm/data/nuscenes_infos_train_temporal_v3_scene.pkl EXISTS
/mnt/data2/whz/dome-cfm/data/nuscenes EXISTS
CAN_LOAD_REAL_DATA = True


## 3. 取一个真实 batch，查看 input_occs / target_occs / metas

如果本机没有 nuScenes 数据，这个 cell 会自动跳过，并在后面用 synthetic metas 继续演示 joint flow。

In [4]:
import torch

input_occs = None
target_occs = None
metas = None

if CAN_LOAD_REAL_DATA:
    import model  # 注册模型，不直接用
    from dataset import get_dataloader

    train_loader, _ = get_dataloader(
        cfg.train_dataset_config,
        cfg.val_dataset_config,
        cfg.train_wrapper_config,
        cfg.val_wrapper_config,
        cfg.train_loader,
        cfg.val_loader,
        dist=False,
    )
    input_occs, target_occs, metas = next(iter(train_loader))
    print('input_occs:', tuple(input_occs.shape), input_occs.dtype)
    print('target_occs:', tuple(target_occs.shape), target_occs.dtype)
    print('metas type:', type(metas), 'len:', len(metas))
    print('metas[0] keys:', sorted(metas[0].keys()))
else:
    print('跳过真实 dataloader：缺少数据文件。')

input_occs: (2, 11, 200, 200, 16) torch.int64
target_occs: (2, 11, 200, 200, 16) torch.int64
metas type: <class 'list'> len: 2
metas[0] keys: ['e2g_r', 'e2g_rel0_r', 'e2g_rel0_t', 'e2g_t', 'gt_mode', 'rel_poses', 'rel_poses_yaws', 'scene_token', 'traj_mode']


## 4. 看 metas 里几个关键字段

普通 non-resample dataset 里通常会有：

- `rel_poses`: 轨迹点，shape `(11, 2)`。
- `gt_mode`: 每帧 mode，shape `(11, 3)`。
- `traj_mode`: 整段主 mode，一个整数。
- `e2g_t/e2g_r`: ego 在 global 坐标里的位置和朝向。

In [5]:
import numpy as np

if metas is not None:
    m = metas[0]
    for key in ['rel_poses', 'gt_mode', 'traj_mode', 'e2g_t', 'e2g_r', 'e2g_rel0_t', 'rel_poses_yaws']:
        if key in m:
            arr = np.asarray(m[key])
            print(f'{key:15s}', 'shape =', arr.shape, 'value/head =')
            print(arr[:3] if arr.ndim > 0 else arr)
            print()
else:
    print('没有真实 metas，跳过。')

rel_poses       shape = (11, 2) value/head =
[[-6.2294748e-05 -7.7796947e-05]
 [-1.0474642e-04 -1.1857620e-04]
 [ 1.9888292e-04 -9.3521761e-05]]

gt_mode         shape = (11, 3) value/head =
[[0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]]

traj_mode       shape = () value/head =
2

e2g_t           shape = (11, 3) value/head =
[[1211.60351453  948.98003174    0.        ]
 [1211.60351476  948.98002974    0.        ]
 [1211.60351553  948.98002856    0.        ]]

e2g_r           shape = (11, 4) value/head =
[[-0.90197598 -0.00782814  0.00203689 -0.43171045]
 [-0.90197367 -0.00782394  0.00206141 -0.43171522]
 [-0.90197451 -0.00781075  0.00210246 -0.43171351]]

e2g_rel0_t      shape = (11, 3) value/head =
[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-1.41259779e-06 -1.44190512e-06  3.25686361e-08]
 [-1.85015484e-06 -2.77754538e-06  5.36490074e-08]]

rel_poses_yaws  shape = (11, 3) value/head =
[[ 1.44190511e-06 -1.41259784e-06  1.09989149e-05]
 [ 1.33563600e-06 -4.37571089e-07 -3.11332894e-06]

## 5. 从 metas 抽 future trajectory 和 command

joint flow 用的是：

```text
traj_start = metas['rel_poses'][4:10, :2]  -> (B, 6, 2)
command    = traj_mode / gt_mode           -> (B,)
```

如果没有真实 metas，下面会造一个 synthetic metas。

In [6]:
from utils.trajectory_condition import extract_commands_from_metas, extract_trajectory_from_metas

if metas is None:
    B = 2
    synthetic_rel_poses = np.zeros((11, 2), dtype=np.float32)
    synthetic_rel_poses[:, 0] = np.linspace(0, 5, 11)
    synthetic_gt_mode = np.zeros((11, 3), dtype=np.float32)
    synthetic_gt_mode[:, 2] = 1  # straight
    metas = [
        {'rel_poses': synthetic_rel_poses.copy(), 'gt_mode': synthetic_gt_mode.copy(), 'traj_mode': 2}
        for _ in range(B)
    ]
else:
    B = len(metas)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

traj_start = extract_trajectory_from_metas(
    metas,
    key=cfg.sample.traj_key,
    start_index=cfg.sample.traj_start_index,
    traj_len=cfg.sample.traj_len,
    traj_dim=cfg.sample.traj_dim,
    device=device,
)
commands = extract_commands_from_metas(
    metas,
    traj=traj_start,
    num_modes=cfg.sample.num_command_modes,
    device=device,
)

print('traj_start:', tuple(traj_start.shape), traj_start.dtype)
print(traj_start[0])
print('commands:', tuple(commands.shape), commands.dtype, commands.tolist())

traj_start: (2, 6, 2) torch.float32
tensor([[ 3.9761e-05, -4.4084e-05],
        [-2.8592e-05, -2.8181e-04],
        [ 1.6132e-04, -2.2393e-04],
        [-2.8708e-04, -1.0601e-04],
        [-1.3146e-04,  3.0250e-04],
        [-2.2000e-04,  2.1309e-04]], device='cuda:0')
commands: (2,) torch.int64 [2, 2]


## 6. 可选：用 OccVAE 编码真实 occupancy

如果有 `ckpts/occvae_latest.pth` 和 CUDA，可以把真实 `input_occs` 编成：

```text
x_start_occ: (B, 11, 64, 25, 25)
```

如果没有 checkpoint 或数据，就跳过，后面用 dummy latent。

In [7]:
from einops import rearrange

x_start_occ = None
vae_ckpt = PROJECT_ROOT / 'ckpts' / 'occvae_latest.pth'

if input_occs is not None and vae_ckpt.exists() and torch.cuda.is_available():
    import model
    from mmengine.registry import MODELS

    vae = MODELS.build(cfg.model.vae).cuda().eval()
    ckpt = torch.load(str(vae_ckpt), map_location='cpu')
    state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
    print(vae.load_state_dict(state_dict, strict=False))

    with torch.no_grad():
        x = input_occs.cuda()
        bs = x.shape[0]
        z, shapes = vae.forward_encoder(x)
        z, _, _ = vae.sample_z(z)
        z = z * cfg.model.vae.scaling_factor
        if z.dim() == 4:
            x_start_occ = rearrange(z, '(b f) c h w -> b f c h w', b=bs).contiguous()
        else:
            x_start_occ = rearrange(z, 'b c f h w -> b f c h w', b=bs).contiguous()

    print('x_start_occ from VAE:', tuple(x_start_occ.shape), x_start_occ.dtype)
else:
    print('跳过 VAE 编码。原因可能是：无真实数据 / 无 ckpt / 无 CUDA。')

[*] Enc has Attn at i_level, i_block: 2, 0
[*] Enc has Attn at i_level, i_block: 2, 1
Working with z of shape (1, 64, 25, 25, 25) = 1000000 dimensions.
[*] Dec has Attn at i_level, i_block: 2, 0
[*] Dec has Attn at i_level, i_block: 2, 1
<All keys matched successfully>


/mnt/data2/congfeng/miniconda3/envs/torchcfm/lib/python3.8/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at /opt/conda/conda-bld/pytorch_1682343998658/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


x_start_occ from VAE: (2, 11, 64, 25, 25) torch.float32


## 7. 构造 tiny JointDome，先跑通 joint flow loss

这个 tiny 模型只用来验证 joint flow 的输入输出 shape，不代表真实 DOME 训练规模。真实模型是 768 hidden / 28 layers。

In [11]:
import model
from mmengine.registry import MODELS
from diffusion import create_joint_flow_matching

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
B = len(metas)

# 用小 latent 跑 shape，避免 notebook 太吃显存。
tiny_x_start_occ = torch.randn(B, 11, 4, 4, 4, device=device)

tiny_model_cfg = dict(
    type='JointDome',
    input_size=4,
    patch_size=1,
    in_channels=4,
    hidden_size=64,
    depth=2,
    num_heads=4,
    num_frames=11,
    learn_sigma=False,
    extras=1,
    attention_mode='math',
    traj_dim=2,
    traj_len=6,
    num_command_modes=3,
    trajectory_encoder=dict(
        type='PoseEncoder_fourier',
        in_channels=2,
        out_channels=64,
        num_layers=2,
        num_modes=3,
        num_fut_ts=1,
        do_proj=True,
        max_length=6,
    ),
    planning_decoder=dict(
        type='PoseDecoder',
        in_channels=64,
        num_layers=2,
        num_modes=3,
        num_fut_ts=6,
        out_dim=2,
    ),
)

tiny_model = MODELS.build(tiny_model_cfg).to(device).train()
joint_flow = create_joint_flow_matching(
    num_sampling_steps=4,
    replace_cond_frames=True,
    cond_frames_choices=[[], [0], [0, 1], [0, 1, 2], [0, 1, 2, 3]],
    model_time_scale=1000.0,
    traj_key='rel_poses',
    traj_start_index=4,
    traj_len=6,
    traj_dim=2,
    traj_loss_weight=10.0,
    num_command_modes=3,
)

loss_dict = joint_flow.training_losses(
    tiny_model,
    tiny_x_start_occ,
    model_kwargs={'metas': metas},
)

print({k: tuple(v.shape) for k, v in loss_dict.items()})
print({k: float(v.mean().detach().cpu()) for k, v in loss_dict.items()})

{'occ_mse': (2,), 'traj_mse': (2,), 'loss': (2,)}
{'occ_mse': 1.519884467124939, 'traj_mse': 1.003714680671692, 'loss': 11.557031631469727}


## 8. 手动拆开 joint flow 的关键中间张量

这一段不调用模型，只看 flow matching 自己构造了什么。

In [9]:
with torch.no_grad():
    x_start = tiny_x_start_occ
    occ_noise = torch.randn_like(x_start)
    traj_start = extract_trajectory_from_metas(
        metas,
        key='rel_poses',
        start_index=4,
        traj_len=6,
        traj_dim=2,
        device=device,
        dtype=x_start.dtype,
    )
    traj_noise = torch.randn_like(traj_start)
    t = torch.rand(B, device=device, dtype=x_start.dtype)

    x_t_occ, u_occ = joint_flow._linear_flow(occ_noise, x_start, t)
    traj_t, u_traj = joint_flow._linear_flow(traj_noise, traj_start, t)
    commands = extract_commands_from_metas(metas, traj=traj_start, num_modes=3, device=device)

print('t:', tuple(t.shape), t)
print('x_t_occ:', tuple(x_t_occ.shape), 'u_occ:', tuple(u_occ.shape))
print('traj_start:', tuple(traj_start.shape))
print('traj_t:', tuple(traj_t.shape), 'u_traj:', tuple(u_traj.shape))
print('commands:', commands.tolist())

t: (2,) tensor([0.9103, 0.0803], device='cuda:0')
x_t_occ: (2, 11, 4, 4, 4) u_occ: (2, 11, 4, 4, 4)
traj_start: (2, 6, 2)
traj_t: (2, 6, 2) u_traj: (2, 6, 2)
commands: [2, 2]


## 9. 直接看 tiny JointDome 的输出 shape

In [10]:
with torch.no_grad():
    out = tiny_model(
        x_t_occ,
        t * 1000.0,
        traj_t=traj_t,
        commands=commands,
        metas=metas,
    )

for k, v in out.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v))

print('\n含义：')
print('occ        -> predicted occupancy velocity')
print('traj_modes -> predicted trajectory velocity for 3 modes')
print('traj       -> command-selected trajectory velocity')

occ (2, 11, 4, 4, 4) torch.float16
traj (2, 6, 2) torch.float16
traj_modes (2, 3, 6, 2) torch.float16
commands (2,) torch.int64

含义：
occ        -> predicted occupancy velocity
traj_modes -> predicted trajectory velocity for 3 modes
traj       -> command-selected trajectory velocity


## 10. 下一步你可以改哪里

- 如果想用第一帧 `gt_mode` 当 command，而不是 `traj_mode`，改 `utils/trajectory_condition.py` 的 `extract_commands_from_metas()`。
- 如果想让 trajectory target 用 `rel_poses_yaws`，把 config 里的 `traj_key='rel_poses_yaws'`，并把 `traj_dim=3`。
- 如果想跑真实训练，用：

```bash
bash tools/train_dome_joint_flow_resample.sh
```

但如果不要 resample，建议再单独做一个 non-resample joint config，把 `train_dataset_config.type` 固定成 `nuScenesSceneDatasetLidar`。